# Quality Window

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

In [ ]:
X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_2D.csv", index_col=[0, 1, 2])
Delaunay = pd.read_csv("../../_temp/v1/delaunay.Xpred_2D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

## GPR (deterministic)

In [ ]:
pred_gpr = pd.read_csv("../../benchmarks/v1/gpr.Xpred_2D.csv").pivot(
    index=["index", "batch"],
    columns="target",
)

### H

In [ ]:
H_THRESHOLD = 1.1

In [ ]:
target = "H"
mean = pred_gpr["latent_mean"][target]
quality_window = mean < H_THRESHOLD
levels = [-0.5, 0.5, 1.5]
cmap = mcolors.ListedColormap(["lightgray", "tab:blue"])
norm = mcolors.BoundaryNorm(levels, cmap.N)

fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred_df = Xpred[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_quality_window = quality_window[ok_pred.values].to_numpy(dtype=int)

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_quality_window.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal", ticks=[0, 1])
cbar.set_ticklabels([f"H ≥ {H_THRESHOLD:g}", f"H < {H_THRESHOLD:g}"])
cbar.set_label("H quality window", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

### phi

In [ ]:
phi_THRESHOLD = 0.25

In [ ]:
target = "phi_1"
mean = pred_gpr["latent_mean"][target]
quality_window = mean < phi_THRESHOLD
levels = [-0.5, 0.5, 1.5]
cmap = mcolors.ListedColormap(["lightgray", "tab:blue"])
norm = mcolors.BoundaryNorm(levels, cmap.N)

fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred_df = Xpred[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_quality_window = quality_window[ok_pred.values].to_numpy(dtype=int)

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_quality_window.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal", ticks=[0, 1])
cbar.set_ticklabels([f"φ ≥ {phi_THRESHOLD:g}", f"φ < {phi_THRESHOLD:g}"])
cbar.set_label("φ quality window", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

### Combined

In [ ]:
H_mean = pred_gpr["latent_mean"]["H"]
phi_mean = pred_gpr["latent_mean"]["phi_1"]
combined_window = (H_mean < H_THRESHOLD) & (phi_mean < phi_THRESHOLD)

levels = [-0.5, 0.5, 1.5]
cmap = mcolors.ListedColormap(["lightgray", "tab:blue"])
norm = mcolors.BoundaryNorm(levels, cmap.N)

fig, axes = plt.subplots(1, len(unique_slurries), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred_df = Xpred[ok_pred]
    this_Xpred = this_Xpred_df.to_xarray().to_array().values
    this_combined_window = combined_window[ok_pred.values].to_numpy(dtype=int)

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])

    contour = ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_combined_window.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    (cos_theta,) = this_Xpred_df["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal", ticks=[0, 1])
cbar.set_ticklabels(
    [
        "Outside combined window",
        f"H < {H_THRESHOLD:g} and φ < {phi_THRESHOLD:g}",
    ]
)
cbar.set_label("Combined quality window", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()

## GPR (Probabilistic)

## GPQR

In [ ]:
marginal = pd.read_csv(
    "../../benchmarks/v1/gpqr.marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)

## Plot

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

## Marginal probability (H)

In [ ]:
marginal_H = marginal[marginal["target"] == "H"].drop(columns=["target"])
marginal_H = marginal_H.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal_H[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (H)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Marginal probability (phi_1)

In [ ]:
marginal_phi = marginal[marginal["target"] == "phi_1"].drop(columns=["target"])
marginal_phi = marginal_phi.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal_phi[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (phi_1)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Joint probability

In [ ]:
joint = pd.read_csv(
    "../../benchmarks/v1/gpqr.joint_probability.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
joint = joint.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = joint[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Joint probability", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Uncertainty of joint probability

In [ ]:
Xpred = pd.read_csv("../../_temp/v1/Xpred_1D.csv", index_col=[0, 1, 2])

joint = pd.read_csv(
    "../../benchmarks/v1/gpqr.joint_probability.Xpred_1D.csv",
    index_col=["index", "sample"],
).drop(columns=["batch"])
joint_mean = joint.groupby(level=["index"]).mean()
joint_interval = joint.groupby(level=["index"]).quantile([0.025, 0.975])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

Rgt_pred = Xpred["gap_to_thickness_ratio"].unique()
Cas = Xpred["capillary_number"].unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].copy()
    this_Xpred["prediction_index"] = np.flatnonzero(ok_pred)

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        prediction_index = this_Xpred.loc[ok_pred, "prediction_index"]

        prob_mean = joint_mean.loc[prediction_index]
        prob_mean = prob_mean["joint_prob"]
        ax.plot(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            prob_mean,
            color=cmap(norm(ca)),
            label=f"Ca={ca:.3f}",
        )

        prob_interval = joint_interval.loc[prediction_index]
        prob_interval = prob_interval["joint_prob"].unstack(level=-1)
        ax.fill_between(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            prob_interval[0.025],
            prob_interval[0.975],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Quantiles")